## Bivariate EMD: does cNF preserve joint structure that the marginal eval misses?

All headline EMDs in the paper are univariate (1D per target variable, then averaged). A reviewer point: this would not catch a model that gets each marginal right but the joint distribution wrong. Here we compute 2D EMD on policy-relevant variable pairs, comparing cNF and the oversampling baseline against the held-out truth.

We use sliced Wasserstein (mean of 1D EMDs over `n_proj` random directions in the 2D plane) as a tractable approximation of the 2D Wasserstein-1 distance. Same noise-floor protocol as the main pipeline (size-matched candidate samples, averaged over `n_subsamples` draws).

All three streams (truth, cNF, train-bootstrap) are loaded from the SCALED per-dataset files emitted by the main pipeline -- they are in the same units, so EMD numbers are directly comparable across datasets after IQR scaling.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, wilcoxon

# Per-dataset config; mirrors the per-dataset evaluate.ipynb conventions.
# 'code' is the 3-letter prefix used in the 'full_{code}_scaled_*' filename.
# 'train_percent' is the TRAIN_PERCENT setting (8 for most, 80 for Yemen).
EXP_BASE = "/data/shared/fsibilla/clean_code/Q1/experiments"
OUT_DIR  = "/data/shared/fsibilla/clean_code/Q1/across_experiments_eval"

DATASETS = {
    "lka_vam": {
        "code": "lka", "train_percent": 8, "adm1_col": "adm1name",
        "pairs": [("FCS", "rCSI"), ("education_score", "log_income")],
    },
    "moz_vam": {
        "code": "moz", "train_percent": 8, "adm1_col": "adm1name",
        "pairs": [("FCS", "rCSI")],
    },
    "yem_mvam": {
        "code": "yem", "train_percent": 80, "adm1_col": "adm1name",
        "pairs": [("FCS", "rCSI")],
    },
}

SEEDS         = [1, 2, 3, 4, 5]
N_PROJ        = 64
N_SUBSAMPLES  = 20

In [ ]:
def _seed_dir(ds_root, tp, seed):
    return os.path.join(ds_root, "results", f"train_{tp}_scaled", f"seed_{seed}_scaled")

def path_full(ds_root, code, tp, seed):
    return os.path.join(_seed_dir(ds_root, tp, seed),
                        f"full_{code}_scaled_train{tp}_seed{seed}.csv")

def path_syn(ds_root, tp, seed):
    return os.path.join(_seed_dir(ds_root, tp, seed),
                        f"generated_pool_{tp}_seed{seed}_scaled.csv")

def path_train(ds_root, tp, seed):
    return os.path.join(_seed_dir(ds_root, tp, seed),
                        f"train_subset_{tp}_seed{seed}_scaled.csv")

def _finite_2d(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.ndim == 1: arr = arr.reshape(-1, 1)
    return arr[np.all(np.isfinite(arr), axis=1)]

def sliced_w1_2d(x_ref, x_cand, n_proj=64, rng=None):
    if rng is None: rng = np.random.default_rng(0)
    xr = _finite_2d(x_ref); xc = _finite_2d(x_cand)
    if len(xr) == 0 or len(xc) == 0: return np.nan
    thetas = rng.uniform(0, 2 * np.pi, size=n_proj)
    dirs = np.stack([np.cos(thetas), np.sin(thetas)], axis=1)
    proj_ref  = xr @ dirs.T
    proj_cand = xc @ dirs.T
    vals = np.empty(n_proj)
    for k in range(n_proj):
        vals[k] = wasserstein_distance(proj_ref[:, k], proj_cand[:, k])
    return float(vals.mean())

def sliced_w1_2d_match(x_ref, x_cand, n_proj=64, n_subsamples=20, rng=None):
    if rng is None: rng = np.random.default_rng(0)
    xr = _finite_2d(x_ref); xc = _finite_2d(x_cand)
    if len(xr) == 0 or len(xc) == 0: return np.nan
    n = len(xr)
    if len(xc) <= n:
        return sliced_w1_2d(xr, xc, n_proj=n_proj, rng=rng)
    vals = []
    for _ in range(n_subsamples):
        idx = rng.choice(len(xc), size=n, replace=False)
        vals.append(sliced_w1_2d(xr, xc[idx], n_proj=n_proj, rng=rng))
    return float(np.nanmean(vals))

def bootstrap_oversample(x_train_2d, n, rng):
    if n <= 0 or len(x_train_2d) == 0:
        return np.empty((0, 2))
    idx = rng.choice(len(x_train_2d), size=n, replace=True)
    return x_train_2d[idx]

In [ ]:
rows = []
for ds_name, cfg in DATASETS.items():
    ds_root = os.path.join(EXP_BASE, ds_name)
    code, tp, adm1_col = cfg["code"], cfg["train_percent"], cfg["adm1_col"]
    print(f"\n==== {ds_name} (code={code}, tp={tp}) ====")
    for v1, v2 in cfg["pairs"]:
        print(f"  pair: ({v1}, {v2})")
        for seed in SEEDS:
            for label, path_fn in [("full", path_full), ("syn", path_syn), ("train", path_train)]:
                p = path_fn(ds_root, code, tp, seed) if label == "full" else path_fn(ds_root, tp, seed)
                if not os.path.exists(p):
                    print(f"    seed {seed} {label}: missing {p}")
                    break
            else:
                full  = pd.read_csv(path_full(ds_root,  code, tp, seed))
                syn   = pd.read_csv(path_syn(ds_root,   tp,   seed))
                trn   = pd.read_csv(path_train(ds_root, tp,   seed))
                # column sanity
                missing = [c for c in [v1, v2, adm1_col] if c not in full.columns]
                if missing:
                    print(f"    seed {seed}: missing cols in full {missing}")
                    continue
                adm1s = sorted(full[adm1_col].dropna().astype(str).unique().tolist())
                for adm1 in adm1s:
                    t   = _finite_2d(full.loc[full[adm1_col] == adm1, [v1, v2]].values)
                    g   = _finite_2d(syn.loc[syn[adm1_col]   == adm1, [v1, v2]].values) if adm1_col in syn.columns else None
                    tra = _finite_2d(trn.loc[trn[adm1_col]   == adm1, [v1, v2]].values) if adm1_col in trn.columns else None
                    if g is None or tra is None or len(t) == 0 or len(g) == 0:
                        continue
                    rng_boot  = np.random.default_rng(seed * 991  + (hash(adm1) % (2**31)))
                    rng_match = np.random.default_rng(seed * 7919 + (hash(adm1) * 31 % (2**31)))
                    over = bootstrap_oversample(tra, len(t), rng_boot)
                    emd_gen  = sliced_w1_2d_match(t, g,    n_proj=N_PROJ, n_subsamples=N_SUBSAMPLES, rng=rng_match)
                    emd_over = sliced_w1_2d(t, over,       n_proj=N_PROJ, rng=rng_match)
                    rows.append({
                        "dataset": ds_name, "pair": f"{v1}+{v2}",
                        "seed": seed, "adm1": adm1,
                        "n_true": len(t), "n_gen": len(g), "n_over": len(over),
                        "emd_gen_2d":  emd_gen,
                        "emd_over_2d": emd_over,
                        "imp_2d":      emd_over - emd_gen,
                    })

results = pd.DataFrame(rows)
print(f"\nTotal rows: {len(results)}")
results.head()

In [ ]:
def cluster_bootstrap_ci(values, pair_ids, n_boot=2000, seed=42, ci=95):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    pair_ids = np.asarray(pair_ids)
    uniq = np.unique(pair_ids)
    idx_map = {p: np.where(pair_ids == p)[0] for p in uniq}
    boots = []
    for _ in range(n_boot):
        sampled = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_map[p] for p in sampled])
        boots.append(np.nanmean(values[idx]))
    lo_q = (100 - ci) / 2
    return float(np.nanmean(values)), float(np.nanpercentile(boots, lo_q)), float(np.nanpercentile(boots, 100 - lo_q))

summary_rows = []
for (ds, pair), sub in results.groupby(["dataset", "pair"]):
    pair_id = sub["seed"].astype(str) + "|" + sub["adm1"].astype(str)
    mean_imp, lo, hi = cluster_bootstrap_ci(sub["imp_2d"].values, pair_id.values)
    mean_gen  = float(np.nanmean(sub["emd_gen_2d"]))
    mean_over = float(np.nanmean(sub["emd_over_2d"]))
    pct = 100 * (mean_over - mean_gen) / mean_over if mean_over > 0 else np.nan
    diffs = sub["emd_over_2d"].values - sub["emd_gen_2d"].values
    diffs = diffs[np.isfinite(diffs)]
    try:
        _stat, p = wilcoxon(diffs, alternative="greater")
    except Exception:
        p = np.nan
    summary_rows.append({
        "dataset": ds, "pair": pair, "n_obs": len(sub),
        "emd_over_2d": mean_over, "emd_gen_2d": mean_gen,
        "abs_improv": mean_imp, "abs_improv_lo": lo, "abs_improv_hi": hi,
        "pct_improv": pct,
        "wilcoxon_p": p,
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUT_DIR, "bivariate_emd_summary.csv"), index=False)
print(f"Saved {os.path.join(OUT_DIR, 'bivariate_emd_summary.csv')}")
summary.round(4)